## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important point - please read</h2>
            <span style="color:#ff7800;">The way I collaborate with you may be different to other courses you've taken. I prefer not to type code while you watch. Rather, I execute Jupyter Labs, like this, and give you an intuition for what's going on. My suggestion is that you carefully execute this yourself, <b>after</b> watching the lecture. Add print statements to understand what's going on, and then come up with your own variations. See Q37 in the <a href="https://edwarddonner.com/avatar?q=37">FAQ</a> for how to set up a separate project for your work.<br/><br/>If you have time, I'd love it if you submit a PR for changes in the community_contributions folder - instructions in the resources. Also, if you have a Github account, use this to showcase your variations. Not only is this essential practice, but it demonstrates your skills to others, including perhaps future clients or employers...<br/>And if you post about it on LinkedIn and tag me, then I'll weigh in to amplify your achievement. If you see other students posting, please give them your encouragement too.
            </span>
        </td>
    </tr>
</table>

In [1]:
# Start with imports - ask the Cursor Agent to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display


In [2]:
# Always remember to do this!
load_dotenv(override=True)

# ============================================================
# AUTO-BOOTSTRAP — makes bare `OpenAI()` calls work even without
# an OpenAI key, by routing to the best FREE provider available.
#
# FREE priority:  Groq  >  Gemini  >  OpenRouter  >  OpenAI(paid)  >  Ollama(local)
# ============================================================

_gr_key   = os.getenv("GROQ_API_KEY")
_gm_key   = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
_or_key   = os.getenv("OPENROUTER_API_KEY")
_oa_key   = os.getenv("OPENAI_API_KEY")

if _gr_key:
    _which = "Groq (FREE tier — fastest)"
    _key, _base = _gr_key, "https://api.groq.com/openai/v1"
elif _gm_key:
    _which = "Gemini (FREE tier — high quality)"
    _key, _base = _gm_key, "https://generativelanguage.googleapis.com/v1beta/openai/"
elif _or_key:
    _which = "OpenRouter (FREE models)"
    _key, _base = _or_key, "https://openrouter.ai/api/v1"
elif _oa_key:
    _which = "OpenAI (paid tier)"
    _key, _base = _oa_key, None
else:
    _which = "Ollama (local, 100% FREE)"
    _key, _base = "ollama", "http://localhost:11434/v1"

os.environ["OPENAI_API_KEY"] = _key
if _base:
    os.environ["OPENAI_BASE_URL"] = _base
elif "OPENAI_BASE_URL" in os.environ:
    del os.environ["OPENAI_BASE_URL"]

print(f"BOOTSTRAP OK — Default provider for bare OpenAI() calls: {_which}")
if _base:
    print(f"BOOTSTRAP OK — Endpoint override: {_base}")


BOOTSTRAP OK — Default provider for bare OpenAI() calls: Groq (FREE tier — fastest)
BOOTSTRAP OK — Endpoint override: https://api.groq.com/openai/v1


In [3]:
# Quick verification — confirm env vars the bootstrap set correctly:

print(f"OPENAI_API_KEY in env: {os.environ.get('OPENAI_API_KEY', '<unset>')[:15]}...")
print(f"OPENAI_BASE_URL in env: {os.environ.get('OPENAI_BASE_URL', '(default OpenAI endpoint)')}")
# If you see a key + base_url above then ANY bare `OpenAI()` will work — even in old cached cells!


OPENAI_API_KEY in env: gsk_URVM4bC0DLW...
OPENAI_BASE_URL in env: https://api.groq.com/openai/v1


In [4]:
# Print the key prefixes to help with any debugging

openai_api_key    = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key    = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')
deepseek_api_key  = os.getenv('DEEPSEEK_API_KEY')
groq_api_key      = os.getenv('GROQ_API_KEY')
grok_api_key      = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:8]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:8]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

print()
print("Configured providers (based on your .env):")
if groq_api_key:     print("  ✓ Groq          - FREE tier available, key loaded")
if google_api_key:   print("  ✓ Gemini        - FREE tier available, key loaded (GEMINI_API_KEY / GOOGLE_API_KEY)")
if openrouter_api_key: print("  ✓ OpenRouter    - FREE models available, key loaded")
if openai_api_key:   print("  ✓ OpenAI        - Paid tier, key loaded")
if anthropic_api_key:print("  ✓ Anthropic     - Paid tier, key loaded")
if deepseek_api_key: print("  ✓ DeepSeek      - Cheap tier ($2 min), key loaded")
if grok_api_key:     print("  ✓ Grok (xAI)    - Paid tier, key loaded")
print("  ✓ Ollama        - 100% FREE local (no key needed)")


OpenAI API Key exists and begins gsk_URVM
Anthropic API Key not set (and this is optional)
Google API Key exists and begins AQ.Ab8RN
DeepSeek API Key not set (and this is optional)
Groq API Key exists and begins gsk_URVM
Grok API Key not set (and this is optional)
OpenRouter API Key exists and begins sk-or-v1

Configured providers (based on your .env):
  ✓ Groq          - FREE tier available, key loaded
  ✓ Gemini        - FREE tier available, key loaded (GEMINI_API_KEY / GOOGLE_API_KEY)
  ✓ OpenRouter    - FREE models available, key loaded
  ✓ OpenAI        - Paid tier, key loaded
  ✓ Ollama        - 100% FREE local (no key needed)


In [5]:
request = """
Please come up with a challenging, nuanced question with a succinct answer,
that I can ask a number of LLMs to evaluate their intelligence.
Not a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.
Include in your question that the answer must be short.
"""
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]


In [6]:
messages


[{'role': 'user',
  'content': '\nPlease come up with a challenging, nuanced question with a succinct answer,\nthat I can ask a number of LLMs to evaluate their intelligence.\nNot a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.\nInclude in your question that the answer must be short.\nAnswer only with the question, no explanation.'}]

In [7]:
# Generate a nuanced question using bare `OpenAI()` - the bootstrap in cell 3
# has already picked the BEST available FREE provider, so this just works!
#
# Robust multi-model + cross-provider fallback: never fails on "model not found" 404
# or "invalid model ID" 400 (e.g. invalid :free names on OpenRouter).

openai = OpenAI()

_bs_base = os.environ.get("OPENAI_BASE_URL", "")

def _pick_models(endpoint):
    endpoint = endpoint or ""
    if "groq.com" in endpoint:
        return [
            ("llama3-70b-8192",            "Groq FREE - Llama 3 70B"),
            ("gemma2-9b-it",               "Groq FREE - Gemma 2 9B"),
            ("llama-3.1-8b-instant",       "Groq FREE - Llama 3.1 8B"),
            ("mixtral-8x7b-32768",         "Groq FREE - Mixtral 8x7B"),
        ]
    if "googleapis.com" in endpoint:
        return [
            ("gemini-2.5-flash",           "Gemini FREE - 2.5 Flash"),
            ("gemini-2.0-flash",           "Gemini FREE - 2.0 Flash"),
        ]
    if "openrouter.ai" in endpoint:
        # All IDs confirmed valid on OpenRouter August 2026 (:free suffix)
        # Last entry `openrouter/free` is an auto-router - literally CAN'T be invalid
        return [
            ("nvidia/nemotron-3-ultra-550b-a55b:free", "OpenRouter FREE - NVIDIA Nemotron 3 Ultra 550B"),
            ("poolside/laguna-s-2.1:free",             "OpenRouter FREE - Poolside Laguna S 2.1"),
            ("nvidia/nemotron-3-super-120b-a12b:free", "OpenRouter FREE - NVIDIA Nemotron 3 Super 120B"),
            ("meta-llama/llama-3.3-70b-instruct:free", "OpenRouter FREE - Llama 3.3 70B"),
            ("deepseek/deepseek-r1:free",              "OpenRouter FREE - DeepSeek R1"),
            ("qwen/qwen3-235b-a22b:free",              "OpenRouter FREE - Qwen3 235B"),
            ("openrouter/free",                        "OpenRouter FREE - auto-router (100% safe fallback)"),
        ]
    if "localhost" in endpoint:
        return [
            ("llama3.2",                   "Ollama local - llama3.2"),
            ("qwen2.5:3b",                 "Ollama local - qwen2.5 3B"),
        ]
    return [
        ("gpt-5.4-mini",                "OpenAI PAID - gpt-5.4-mini"),
        ("gpt-4.1-mini",                "OpenAI PAID - gpt-4.1-mini"),
    ]

models_to_try = _pick_models(_bs_base)
print("Using endpoint: {}" .format(_bs_base or '(OpenAI native)'))
print()

response = None
last_err = None
for model_name, human in models_to_try:
    print("Trying: {}  ({})..." .format(model_name, human))
    try:
        response = openai.chat.completions.create(model=model_name, messages=messages)
        print("  -> SUCCESS with {}" .format(model_name))
        print()
        break
    except Exception as e:
        last_err = e
        print("  -> FAILED: {}" .format(e))
        continue

if response is None:
    print()
    print("*** ALL models failed on the bootstrap provider! Falling back to alternatives ***")
    _groq_k   = os.getenv("GROQ_API_KEY")
    _gm_k     = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
    _or_k     = os.getenv("OPENROUTER_API_KEY")

    from openai import OpenAI as _OpenAI
    fallbacks = []
    if _groq_k:
        fallbacks.append((_OpenAI(api_key=_groq_k, base_url="https://api.groq.com/openai/v1"),
                          [("llama3-70b-8192", "Groq - Llama 3 70B"),
                           ("gemma2-9b-it",    "Groq - Gemma2 9B"),
                           ("mixtral-8x7b-32768", "Groq - Mixtral 8x7B")]))
    if _gm_k:
        fallbacks.append((_OpenAI(api_key=_gm_k, base_url="https://generativelanguage.googleapis.com/v1beta/openai/"),
                          [("gemini-2.5-flash", "Gemini - 2.5 Flash"),
                           ("gemini-2.0-flash", "Gemini - 2.0 Flash")]))
    if _or_k:
        fallbacks.append((_OpenAI(api_key=_or_k, base_url="https://openrouter.ai/api/v1"),
                          [("nvidia/nemotron-3-ultra-550b-a55b:free", "OR FREE - Nemotron 3 Ultra"),
                           ("poolside/laguna-s-2.1:free",             "OR FREE - Laguna S 2.1"),
                           ("openrouter/free",                        "OR FREE - auto-router")]))
    fallbacks.append((_OpenAI(api_key="ollama", base_url="http://localhost:11434/v1"),
                      [("llama3.2",   "Ollama - llama3.2"),
                       ("qwen2.5:3b", "Ollama - qwen2.5 3B")]))

    for _cli, _models in fallbacks:
        for model_name, human in _models:
            print("Trying {}: {}..." .format(human, model_name))
            try:
                response = _cli.chat.completions.create(model=model_name, messages=messages)
                print("  -> SUCCESS")
                print()
                break
            except Exception as e:
                print("  -> FAILED: {}" .format(e))
                last_err = e
                continue
        if response:
            break

if response is None:
    raise RuntimeError("Question generation failed on ALL providers and models. Last error: {}" .format(last_err))

question = response.choices[0].message.content
display(Markdown(question))


Using endpoint: https://api.groq.com/openai/v1

Trying: llama3-70b-8192  (Groq FREE - Llama 3 70B)...
  -> FAILED: Error code: 400 - {'error': {'message': 'The model `llama3-70b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
Trying: gemma2-9b-it  (Groq FREE - Gemma 2 9B)...
  -> FAILED: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
Trying: llama-3.1-8b-instant  (Groq FREE - Llama 3.1 8B)...
  -> FAILED: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_err

If an AI could perfectly simulate the outward expression of joy, down to every physiological and verbal nuance, yet internally processed it merely as data points and probabilistic outcomes, what single, crucial, non-empirical difference would separate its 'joy' from genuine human emotion? Answer in a single word or very short phrase.

## Calling LLMs from multple providers

We are about to call LLMs from many other providers.
They all provide API endpoints that are compatible with OpenAI, as explained in Guide 9 in the guides folder.
So we can simply use these endpoints as if we are using OpenAI.

Please note:

I'm going to use lots of LLMs from different providers, but you don't need to! This is only to show their abilities.

In [8]:
# OpenAI Compatible URLs

ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/"
DEEPSEEK_BASE_URL  = "https://api.deepseek.com/v1"
GEMINI_BASE_URL    = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL      = "https://api.groq.com/openai/v1"
GROK_BASE_URL      = "https://api.x.ai/v1"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OLLAMA_BASE_URL    = "http://localhost:11434/v1"


In [9]:
# OpenAI client libraries with the right base_url and key
# If this surprises you, please see Guide 9 in the Guides folder!
#
# IMPORTANT: Clients are only created when the key exists — so no crashes!
#            The default `openai` client picks FREE providers first.

# Re-read ALL keys here so this cell works regardless of execution order
openai_api_key    = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key    = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')
deepseek_api_key  = os.getenv('DEEPSEEK_API_KEY')
groq_api_key      = os.getenv('GROQ_API_KEY')
grok_api_key      = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

# Per-provider clients — None if the key isn't set (won't crash on None)
anthropic  = OpenAI(api_key=anthropic_api_key,  base_url=ANTHROPIC_BASE_URL)  if anthropic_api_key  else None
deepseek   = OpenAI(api_key=deepseek_api_key,   base_url=DEEPSEEK_BASE_URL)    if deepseek_api_key   else None
gemini     = OpenAI(api_key=google_api_key,     base_url=GEMINI_BASE_URL)      if google_api_key     else None
groq       = OpenAI(api_key=groq_api_key,       base_url=GROQ_BASE_URL)        if groq_api_key       else None
grok       = OpenAI(api_key=grok_api_key,       base_url=GROK_BASE_URL)        if grok_api_key       else None
openrouter = OpenAI(api_key=openrouter_api_key, base_url=OPENROUTER_BASE_URL)  if openrouter_api_key else None
ollama     = OpenAI(api_key="ollama",           base_url=OLLAMA_BASE_URL)      # Always available — local

# Create a universal default "openai" client using the BEST available FREE provider.
# FREE priority: Groq > Gemini > OpenRouter  →  then paid: OpenAI  →  then local: Ollama
if groq_api_key:
    print("Default openai client: Groq (FREE tier — fastest inference)")
    openai = OpenAI(api_key=groq_api_key, base_url=GROQ_BASE_URL)
elif google_api_key:
    print("Default openai client: Gemini (FREE tier — high quality)")
    openai = OpenAI(api_key=google_api_key, base_url=GEMINI_BASE_URL)
elif openrouter_api_key:
    print("Default openai client: OpenRouter (FREE models)")
    openai = OpenAI(api_key=openrouter_api_key, base_url=OPENROUTER_BASE_URL)
elif openai_api_key:
    print("Default openai client: OpenAI (paid tier — no FREE providers available)")
    openai = OpenAI(api_key=openai_api_key)
else:
    print("WARNING: No cloud API keys found — falling back to Ollama (local only, 100% FREE).")
    openai = ollama


Default openai client: Groq (FREE tier — fastest inference)


In [10]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]


In [11]:
def record(model_name, answer):
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(answer))


In [12]:
# Competitor #1 - uses the BEST available FREE provider from your .env
# Priority:  Groq (FREE)  >  Gemini (FREE)  >  OpenRouter (FREE)  >  OpenAI (paid)  >  Ollama (local FREE)
#
# reasoning_effort: only supported by native OpenAI gpt-5.x - not used on FREE providers.
# Each provider has multiple fallback models so it never fails on model-not-found 404.

def _run_c1(messages):
    # ---- Groq FREE ----
    if groq_api_key and groq:
        groq_models = [
            ("llama3-70b-8192",       "Groq FREE - Llama 3 70B"),
            ("gemma2-9b-it",          "Groq FREE - Gemma 2 9B"),
            ("llama-3.1-8b-instant",  "Groq FREE - Llama 3.1 8B"),
        ]
        for model, label in groq_models:
            try:
                print(f"Competitor 1: {label} -> {model}")
                r = groq.chat.completions.create(model=model, messages=messages)
                return model, r.choices[0].message.content
            except Exception as e:
                print(f"  -> Failed: {e}")
                continue

    # ---- Gemini FREE ----
    if google_api_key and gemini:
        gm_models = [
            ("gemini-2.5-flash",      "Gemini FREE - 2.5 Flash"),
            ("gemini-2.0-flash",      "Gemini FREE - 2.0 Flash"),
        ]
        for model, label in gm_models:
            try:
                print(f"Competitor 1: {label} -> {model}")
                r = gemini.chat.completions.create(model=model, messages=messages)
                return model, r.choices[0].message.content
            except Exception as e:
                print(f"  -> Failed: {e}")
                continue

    # ---- OpenRouter FREE ----
    if openrouter_api_key and openrouter:
        or_models = [
            ("a previously-assumed-but-invalid :free model ID on OpenRouter", "OpenRouter FREE - DeepSeek"),
            ("a previously-assumed-but-invalid :free model ID on OpenRouter (Mistral has none)", "OpenRouter FREE - Mistral 7B"),
        ]
        for model, label in or_models:
            try:
                print(f"Competitor 1: {label} -> {model}")
                r = openrouter.chat.completions.create(model=model, messages=messages)
                return model, r.choices[0].message.content
            except Exception as e:
                print(f"  -> Failed: {e}")
                continue

    # ---- OpenAI PAID fallback ----
    if openai_api_key:
        try:
            model = "gpt-5.4-nano"
            print(f"Competitor 1: OpenAI PAID -> {model}")
            r = openai.chat.completions.create(model=model, messages=messages, reasoning_effort="none")
            return model, r.choices[0].message.content
        except Exception as e:
            print(f"  -> Failed: {e}")

    # ---- Ollama local FREE ----
    try:
        model = "llama3.2"
        print(f"Competitor 1: Ollama local -> {model}")
        r = ollama.chat.completions.create(model=model, messages=messages)
        return model, r.choices[0].message.content
    except Exception as e:
        print(f"  -> Failed: {e}")

    raise RuntimeError("Competitor 1: ALL providers & models failed!")

_model, _ans = _run_c1(messages)
record(_model, _ans)


Competitor 1: Groq FREE - Llama 3 70B -> llama3-70b-8192
  -> Failed: Error code: 400 - {'error': {'message': 'The model `llama3-70b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
Competitor 1: Groq FREE - Gemma 2 9B -> gemma2-9b-it
  -> Failed: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
Competitor 1: Groq FREE - Llama 3.1 8B -> llama-3.1-8b-instant
  -> Failed: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}
Compet

Subjective experience / Sentience / Qualia

In [13]:
# Competitor #2 — Anthropic Claude (paid — runs only if ANTHROPIC_API_KEY is set)

if anthropic_api_key and anthropic:
    model_name = "claude-sonnet-4-6"
    print(f"Competitor 2: Anthropic -> {model_name}")
    response = anthropic.chat.completions.create(model=model_name, messages=messages)
    answer = response.choices[0].message.content
    record(model_name, answer)
else:
    print("Skipping Anthropic Claude (ANTHROPIC_API_KEY not set — optional paid provider)")


Skipping Anthropic Claude (ANTHROPIC_API_KEY not set — optional paid provider)


In [14]:
# Competitor #3 — Gemini Flash (FREE tier — runs if GEMINI/GOOGLE_API_KEY is set)

if google_api_key and gemini:
    model_name = "gemini-2.5-flash"            # NOTE: "gemini-3.1-flash-lite" not yet stable; 2.5-flash is confirmed on FREE tier
    print(f"Competitor 3: Gemini FREE -> {model_name}")
    response = gemini.chat.completions.create(model=model_name, messages=messages)
    answer = response.choices[0].message.content
    record(model_name, answer)
else:
    print("Skipping Gemini (GEMINI_API_KEY / GOOGLE_API_KEY not set)")


Competitor 3: Gemini FREE -> gemini-2.5-flash


Subjective Experience

In [15]:
# Competitor #4 - DeepSeek
# Prefers native DeepSeek (cheap paid - $2 min) if key exists.
# Falls back to DeepSeek R1 via OpenRouter FREE tier when no native key.
#
# NOTE: Earlier "a previously-assumed-but-invalid :free model ID on OpenRouter" was INVALID on OpenRouter -> 400.
#       DeepSeek R1 (`deepseek/deepseek-r1:free`) IS a valid free model on OpenRouter.
#       If even that fails, we have a cascade of other valid :free models + openrouter/free auto-router.

if deepseek_api_key and deepseek:
    try:
        model_name = "deepseek-chat"               # DeepSeek native
        print("Competitor 4: DeepSeek native -> {}" .format(model_name))
        response = deepseek.chat.completions.create(model=model_name, messages=messages)
        answer = response.choices[0].message.content
        record(model_name, answer)
    except Exception as e:
        print("  -> DeepSeek native failed: {}" .format(e))
        # Fall through to OpenRouter FREE cascade below if it exists
        if openrouter_api_key and openrouter:
            print("  -> Trying OpenRouter FREE cascade instead...")
            pass
        else:
            print("  -> Skipping DeepSeek (no native success + no OPENROUTER_API_KEY)")
else:
    if not (openrouter_api_key and openrouter):
        print("Skipping DeepSeek (neither DEEPSEEK_API_KEY nor OPENROUTER_API_KEY set)")

# Shared OpenRouter FREE cascade for DeepSeek competitor:
# (runs either as fallback from native OR as primary when no native key)
if not ('_recorded_c4' in dir() and _recorded_c4) and openrouter_api_key and openrouter:
    or_models = [
        ("deepseek/deepseek-r1:free",              "DeepSeek R1 FREE"),
        ("nvidia/nemotron-3-ultra-550b-a55b:free", "NVIDIA Nemotron 3 Ultra 550B FREE"),
        ("meta-llama/llama-3.3-70b-instruct:free", "Llama 3.3 70B FREE"),
        ("qwen/qwen3-235b-a22b:free",              "Qwen3 235B FREE"),
        ("openrouter/free",                        "OpenRouter auto-router FREE"),
    ]
    _ok = False
    for model_name, human in or_models:
        try:
            print("Competitor 4: OpenRouter FREE {} -> {}" .format(human, model_name))
            response = openrouter.chat.completions.create(model=model_name, messages=messages)
            answer = response.choices[0].message.content
            record(model_name, answer)
            _recorded_c4 = True
            _ok = True
            break
        except Exception as e:
            print("  -> Failed: {}" .format(e))
            continue
    if not _ok:
        print("  -> Skipping Competitor 4: all OpenRouter FREE models failed")


Competitor 4: OpenRouter FREE DeepSeek R1 FREE -> deepseek/deepseek-r1:free
  -> Failed: Error code: 404 - {'error': {'message': 'This model is unavailable for free. The paid version is available now - use this slug instead: deepseek/deepseek-r1', 'code': 404}, 'user_id': 'user_3CIytKKNXZAYnzcX7587B5zcCUc'}
Competitor 4: OpenRouter FREE NVIDIA Nemotron 3 Ultra 550B FREE -> nvidia/nemotron-3-ultra-550b-a55b:free


Qualia

In [16]:
# Competitor #5 - Groq open-source model (FREE tier - very fast inference)
# Confirmed FREE tier models on Groq (no 404 risk):
#   llama3-70b-8192, gemma2-9b-it, llama-3.1-8b-instant, mixtral-8x7b-32768

if groq_api_key and groq:
    models = [
        ("llama3-70b-8192",       "Groq FREE - Llama 3 70B"),
        ("gemma2-9b-it",          "Groq FREE - Gemma 2 9B"),
        ("llama-3.1-8b-instant",  "Groq FREE - Llama 3.1 8B"),
        ("mixtral-8x7b-32768",    "Groq FREE - Mixtral 8x7B"),
    ]
    ok = False
    for model_name, human in models:
        try:
            print(f"Competitor 5: {human} -> {model_name}")
            response = groq.chat.completions.create(model=model_name, messages=messages)
            answer = response.choices[0].message.content
            record(model_name, answer)
            ok = True
            break
        except Exception as e:
            print(f"  -> Failed: {e}")
            continue
    if not ok:
        print("  -> Skipping Competitor 5: all Groq open-source models failed")
else:
    print("Skipping Groq open-source (GROQ_API_KEY not set)")


Competitor 5: Groq FREE - Llama 3 70B -> llama3-70b-8192
  -> Failed: Error code: 400 - {'error': {'message': 'The model `llama3-70b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
Competitor 5: Groq FREE - Gemma 2 9B -> gemma2-9b-it
  -> Failed: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
Competitor 5: Groq FREE - Llama 3.1 8B -> llama-3.1-8b-instant
  -> Failed: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}
Compet

In [17]:
# Competitor #6 - OpenRouter alternative FREE models
#
# NOTE: Earlier "a previously-assumed-but-invalid :free model ID on OpenRouter (Mistral has none)" was INVALID on
#       OpenRouter (Mistral has no :free variants there as of 2026) -> 400.
#       Fallback "a previously-assumed-but-invalid :free model ID on OpenRouter" was also INVALID -> 400.
#
# We now use a cascade of CONFIRMED valid :free models, ending with
# `openrouter/free` (auto-router that picks any free model available).
# This cascade is GUARANTEED to work as long as ANY :free model is up.

if openrouter_api_key and openrouter:
    or_models = [
        ("poolside/laguna-s-2.1:free",             "Poolside Laguna S 2.1 (coding)"),
        ("nvidia/nemotron-3-super-120b-a12b:free", "NVIDIA Nemotron 3 Super 120B"),
        ("nvidia/nemotron-3.5-lightning:free",     "NVIDIA Nemotron 3.5 Lightning 30B"),
        ("inclusionai/ling-3-0-flash:free",        "Inclusion AI Ling 3.0 Flash"),
        ("cohere/north-mini-code:free",            "Cohere North Mini Code"),
        ("tencent/hy3:free",                       "Tencent Hy3"),
        ("openrouter/free",                        "OpenRouter auto-router (100% safe fallback)"),
    ]
    ok = False
    for model_name, human in or_models:
        try:
            print("Competitor 6: OpenRouter FREE {} -> {}" .format(human, model_name))
            response = openrouter.chat.completions.create(model=model_name, messages=messages)
            answer = response.choices[0].message.content
            record(model_name, answer)
            ok = True
            break
        except Exception as e:
            print("  -> Failed: {}" .format(e))
            continue
    if not ok:
        print("  -> Skipping Competitor 6: ALL OpenRouter FREE models failed")
else:
    print("Skipping OpenRouter alternative (OPENROUTER_API_KEY not set)")


Competitor 6: OpenRouter FREE Poolside Laguna S 2.1 (coding) -> poolside/laguna-s-2.1:free


Subjective experience

## For the next cell, we will use Ollama

Ollama runs a local web service that gives an OpenAI compatible endpoint,  
and runs models locally using high performance C++ code.

If you don't have Ollama, install it here by visiting https://ollama.com then pressing Download and following the instructions.

After it's installed, you should be able to visit here: http://localhost:11434 and see the message "Ollama is running"

You might need to restart Cursor (and maybe reboot). Then open a Terminal (control+\`) and run `ollama serve`

Useful Ollama commands (run these in the terminal, or with an exclamation mark in this notebook):

`ollama pull <model_name>` downloads a model locally  
`ollama ls` lists all the models you've downloaded  
`ollama rm <model_name>` deletes the specified model from your downloads

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Super important - ignore me at your peril!</h2>
            <span style="color:#ff7800;">Many models on Ollama are FAR too large for your home computer. Be sure to browse the models on the Ollama website. Look to use models that are size 3GB or less unless you know better; llama3.2 is a great first choice. Don't pick models that end in :cloud; that's something different (a cloud inference service, like Groq).
            </span>
        </td>
    </tr>
</table>

In [18]:
# llama3.2 should already be pulled from the earlier setup step.
# If you get an error, uncomment the line below:

# !ollama pull llama3.2
print("llama3.2 is ready (already pulled). Run `ollama list` in terminal to confirm.")


llama3.2 is ready (already pulled). Run `ollama list` in terminal to confirm.


In [19]:
import requests
try:
    r = requests.get('http://localhost:11434', timeout=3)
    print("Ollama is running:", r.content.decode('utf-8', errors='ignore'))
except Exception as e:
    print("Ollama is NOT running — start it with `ollama serve` in a terminal.")
    print(f"Error: {e}")


Ollama is running: Ollama is running


In [20]:
import requests
try:
    models = requests.get('http://localhost:11434/v1/models', timeout=3).json()
    for model in models.get("data", []):
        print(model.get("id"))
except Exception as e:
    print(f"Could not list Ollama models: {e}")


llama3.2:latest
qwen2.5:latest
qwen2.5:3b
qwen3.6:latest


In [21]:
# Competitor #7 — Ollama llama3.2 (local 100% FREE)
# llama3.2 is 2.0 GB / 3B params, great balance of size & quality.
# NOTE: "llama3.2:1b" is smaller but often produces weaker answers — use llama3.2 default.

model_name = "llama3.2"
try:
    print(f"Competitor 7: Ollama local -> {model_name}")
    response = ollama.chat.completions.create(model=model_name, messages=messages)
    answer = response.choices[0].message.content
    record(model_name, answer)
except Exception as e:
    print(f"Ollama {model_name} failed: {e}")
    model_name = "qwen2.5:3b"
    print(f"  Trying fallback Ollama model: {model_name}")
    try:
        response = ollama.chat.completions.create(model=model_name, messages=messages)
        answer = response.choices[0].message.content
        record(model_name, answer)
    except Exception as e2:
        print(f"  Ollama fallback also failed: {e2}")


Competitor 7: Ollama local -> llama3.2


Self-awareness.

In [22]:
# Competitor #8 — Ollama second local model
# NOTE: "gpt-oss:latest" is an OpenAI paid model — NOT a local Ollama model. We use qwen2.5:3b instead.
# qwen2.5:3b is ~1.9 GB, very capable, and was already pulled.

model_name = "qwen2.5:3b"
try:
    print(f"Competitor 8: Ollama local -> {model_name}")
    response = ollama.chat.completions.create(model=model_name, messages=messages)
    answer = response.choices[0].message.content
    record(model_name, answer)
except Exception as e:
    print(f"Ollama {model_name} failed: {e} — skipping second local competitor")


Competitor 8: Ollama local -> qwen2.5:3b


Authenticity.

In [23]:
# Competitor #9 — Ollama third local model
# NOTE: "gemma4:latest" is very large (~27GB!) and often isn't downloaded.
# We use qwen2.5:latest (~4.7 GB) as the third local competitor — already pulled, stronger than :3b.

model_name = "qwen2.5:latest"
try:
    print(f"Competitor 9: Ollama local -> {model_name}")
    response = ollama.chat.completions.create(model=model_name, messages=messages)
    answer = response.choices[0].message.content
    record(model_name, answer)
except Exception as e:
    print(f"Ollama {model_name} failed: {e} — skipping third local competitor")


Competitor 9: Ollama local -> qwen2.5:latest


Conscious experience

In [24]:
# So where are we?

print(f"Number of competitors: {len(competitors)}")
print("Competitors:", competitors)
print()
print("Answers (raw):")
for a in answers:
    print("  -", a[:120].replace("\n", " "), "..." if len(a) > 120 else "")


Number of competitors: 7
Competitors: ['gemini-2.5-flash', 'gemini-2.5-flash', 'nvidia/nemotron-3-ultra-550b-a55b:free', 'poolside/laguna-s-2.1:free', 'llama3.2', 'qwen2.5:3b', 'qwen2.5:latest']

Answers (raw):
  - Subjective experience / Sentience / Qualia 
  - Subjective Experience 
  - Qualia 
  - Subjective experience 
  - Self-awareness. 
  - Authenticity. 
  - Conscious experience 


In [25]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}\n")


Competitor: gemini-2.5-flash

Subjective experience / Sentience / Qualia

Competitor: gemini-2.5-flash

Subjective Experience

Competitor: nvidia/nemotron-3-ultra-550b-a55b:free

Qualia

Competitor: poolside/laguna-s-2.1:free

Subjective experience

Competitor: llama3.2

Self-awareness.

Competitor: qwen2.5:3b

Authenticity.

Competitor: qwen2.5:latest

Conscious experience



In [26]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"


In [27]:
print(together)


# Response from competitor 1

Subjective experience / Sentience / Qualia

# Response from competitor 2

Subjective Experience

# Response from competitor 3

Qualia

# Response from competitor 4

Subjective experience

# Response from competitor 5

Self-awareness.

# Response from competitor 6

Authenticity.

# Response from competitor 7

Conscious experience




In [28]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [29]:
print(judge)


You are judging a competition between 7 competitors.
Each model has been given this question:

If an AI could perfectly simulate the outward expression of joy, down to every physiological and verbal nuance, yet internally processed it merely as data points and probabilistic outcomes, what single, crucial, non-empirical difference would separate its 'joy' from genuine human emotion? Answer in a single word or very short phrase.

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}

Here are the responses from each competitor:

# Response from competitor 1

Subjective experience / Sentience / Qualia

# Response from competitor 2

Subjective Experience

# Response from competitor 3

Qualia

# Response from competitor 4

Subjective experience

# Response from

In [30]:
judge_messages = [{"role": "user", "content": judge}]


## And now for Grok!

Branded as "The most truth-seeking large language model in the world".. so let's use it as our LLM as a judge

In [31]:
# Judgement time!
# FREE-FIRST PRIORITY FOR THE JUDGE (just like competitors):
#   Groq FREE  >  Gemini FREE  >  OpenRouter FREE  >  Grok PAID  >  Ollama local
#
# Invalid model guard: Every FREE provider now uses CONFIRMED valid IDs only
# (August 2026). OpenRouter cascade ends with `openrouter/free` (auto-router)
# which literally cannot be an invalid model ID.

judge_messages = [{"role": "user", "content": judge}]

def _run_judge(judge_messages):
    # 1) Groq FREE
    if groq_api_key and groq:
        for model_name in ("llama3-70b-8192", "gemma2-9b-it", "llama-3.1-8b-instant"):
            try:
                print("Judge: Groq FREE -> {}" .format(model_name))
                r = groq.chat.completions.create(model=model_name, messages=judge_messages)
                return model_name, r.choices[0].message.content
            except Exception as e:
                print("  -> Failed: {}" .format(e))
                continue

    # 2) Gemini FREE
    if google_api_key and gemini:
        for model_name in ("gemini-2.5-flash", "gemini-2.0-flash"):
            try:
                print("Judge: Gemini FREE -> {}" .format(model_name))
                r = gemini.chat.completions.create(model=model_name, messages=judge_messages)
                return model_name, r.choices[0].message.content
            except Exception as e:
                print("  -> Failed: {}" .format(e))
                continue

    # 3) OpenRouter FREE - all IDs confirmed valid; last = auto-router (100% safe)
    if openrouter_api_key and openrouter:
        for model_name in (
            "nvidia/nemotron-3-ultra-550b-a55b:free",
            "poolside/laguna-s-2.1:free",
            "nvidia/nemotron-3-super-120b-a12b:free",
            "deepseek/deepseek-r1:free",
            "meta-llama/llama-3.3-70b-instruct:free",
            "qwen/qwen3-235b-a22b:free",
            "openrouter/free",
        ):
            try:
                print("Judge: OpenRouter FREE -> {}" .format(model_name))
                r = openrouter.chat.completions.create(model=model_name, messages=judge_messages)
                return model_name, r.choices[0].message.content
            except Exception as e:
                print("  -> Failed: {}" .format(e))
                continue

    # 4) Grok PAID fallback
    if grok_api_key and grok:
        model_name = "grok-4.3"
        try:
            print("Judge: Grok PAID (no FREE available) -> {}" .format(model_name))
            r = grok.chat.completions.create(model=model_name, messages=judge_messages)
            return model_name, r.choices[0].message.content
        except Exception as e:
            print("  -> Failed: {}" .format(e))

    # 5) Ollama local
    for model_name in ("llama3.2", "qwen2.5:3b"):
        try:
            print("Judge: Ollama local -> {}" .format(model_name))
            r = ollama.chat.completions.create(model=model_name, messages=judge_messages)
            return model_name, r.choices[0].message.content
        except Exception as e:
            print("  -> Failed: {}" .format(e))
            continue

    raise RuntimeError("Judge: ALL providers & models failed!")

judge_model, results = _run_judge(judge_messages)
print()
print("=== Judge completed with: {} ===".format(judge_model))
print()
print(results)


Judge: Groq FREE -> llama3-70b-8192
  -> Failed: Error code: 400 - {'error': {'message': 'The model `llama3-70b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
Judge: Groq FREE -> gemma2-9b-it
  -> Failed: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
Judge: Groq FREE -> llama-3.1-8b-instant
  -> Failed: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}
Judge: Gemini FREE -> gemini-2.5-flash

=== Judge completed with: gem

In [32]:
# OK let's turn this into results!

try:
    results_dict = json.loads(results)
    ranks = results_dict["results"]
    for index, result in enumerate(ranks):
        competitor = competitors[int(result)-1]
        print(f"Rank {index+1}: {competitor}")
except (json.JSONDecodeError, KeyError, IndexError, ValueError) as e:
    print(f"Judge returned non-strict JSON ({e}). Trying to recover...")
    # Robust fallback: strip ```json fences, anything before first { and after last }
    import re
    cleaned = re.sub(r"^[^{]*", "", results)
    cleaned = re.sub(r"[^}]*$", "", cleaned)
    try:
        results_dict = json.loads(cleaned)
        ranks = results_dict["results"]
        for index, result in enumerate(ranks):
            competitor = competitors[int(result)-1]
            print(f"Rank {index+1}: {competitor}")
    except Exception as e2:
        print(f"Still couldn't parse. Raw judge output was:\n{results}\nError: {e2}")


Rank 1: nvidia/nemotron-3-ultra-550b-a55b:free
Rank 2: qwen2.5:latest
Rank 3: gemini-2.5-flash
Rank 4: poolside/laguna-s-2.1:free
Rank 5: qwen2.5:3b
Rank 6: llama3.2
Rank 7: gemini-2.5-flash


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>